# Compare OGE's eGRID file to eGRID

Compares the eGRID-format workbook that OGE writes to `results/{year}/eGRID/` with eGRID's
data file for the same year, sheet by sheet and column by column. Both workbooks use
eGRID's column codes, so columns are compared without renaming.

The notebook covers:
1. **Coverage**: which columns are filled in both files, only in one, or in neither
2. **Row matching**: rows that are only in one file
3. **Number columns**: totals and how closely each matched row agrees
4. **Code and text columns**: how often each matched row has the same value
5. **Drill-down**: the rows with the largest differences for one column
6. **Known differences**: the causes of the differences we already know about

eGRID's data files for 2023 and earlier are EPA's, and the files for 2024 and later are
from Cornerstone.

In [ ]:
import sys

import numpy as np
import pandas as pd
from IPython.display import display

%reload_ext autoreload
%autoreload 2

# Tell python where to look for modules.
sys.path.append("../../src")

from oge.filepaths import downloads_folder, results_folder
from oge.output_egrid_file import (
    EGRID_CATEGORIES,
    EGRID_SHEETS,
    get_egrid_columns,
    get_unit,
)

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)

## Load both workbooks

In [ ]:
year = 2024

OGE_FILE = results_folder(f"{year}/eGRID/oge_egrid{year}_data.xlsx")
EGRID_FILE = downloads_folder(f"egrid/egrid{year}_data.xlsx")

In [ ]:
# columns that identify each sheet's rows. The U.S. sheet has a single row
ROW_KEYS = {
    "UNT": ["ORISPL", "UNITID"],
    "GEN": ["ORISPL", "GENID"],
    "PLNT": ["ORISPL"],
    "ST": ["PSTATABB"],
    "BA": ["BACODE"],
    "SRL": ["SUBRGN"],
    "NRL": ["NERC"],
    "US": [],
    "GGL": ["REGION"],
}
# columns that are not compared: the year and the row numbers, which depend on each
# file's rows
SKIPPED_COLUMNS = ["YEAR"] + [
    f"SEQ{sheet}{str(year)[-2:]}" for sheet in ["UNT", "GEN", "PLT"]
]
# relative differences that bin how closely the values of matched rows agree
TOLERANCES = {"≤0.1%": 0.001, "≤1%": 0.01, "≤5%": 0.05, "≤50%": 0.5}


def read_sheet(path: str, sheet: str) -> pd.DataFrame:
    """Reads a sheet of an eGRID workbook, with the codes in row 2 as columns and
    the IDs of units and generators as text."""
    table = pd.read_excel(
        path,
        sheet_name=f"{sheet}{str(year)[-2:]}",
        header=1,
        dtype={"UNITID": str, "GENID": str},
    )
    for key in ROW_KEYS[sheet]:
        if table[key].dtype == object:
            table[key] = table[key].str.strip()
    return table


oge = {sheet: read_sheet(OGE_FILE, sheet) for sheet in EGRID_SHEETS}
egrid = {sheet: read_sheet(EGRID_FILE, sheet) for sheet in EGRID_SHEETS}
pd.DataFrame(
    {
        "OGE rows": {sheet: len(table) for sheet, table in oge.items()},
        "eGRID rows": {sheet: len(table) for sheet, table in egrid.items()},
    }
)

## 1. Coverage

Each column of OGE's spec is `filled in both` files, `only in eGRID` (a gap in OGE's
file), `only in OGE`, or `blank in both`. Columns of eGRID's file that are not in OGE's
spec are listed separately.

In [ ]:
def filled_columns(table: pd.DataFrame) -> set[str]:
    return {column for column in table.columns if table[column].notna().any()}


coverage = []
for sheet in EGRID_SHEETS:
    oge_filled = filled_columns(oge[sheet])
    egrid_filled = filled_columns(egrid[sheet])
    for code, header, category in get_egrid_columns(sheet, year):
        coverage.append(
            {
                "sheet": sheet,
                "code": code,
                "header": header,
                "category": EGRID_CATEGORIES.get(category, "(attributes)"),
                "in_oge": code in oge_filled,
                "in_egrid": code in egrid_filled,
            }
        )
coverage = pd.DataFrame(coverage)
coverage["status"] = np.select(
    [
        coverage["in_oge"] & coverage["in_egrid"],
        coverage["in_egrid"],
        coverage["in_oge"],
    ],
    ["filled in both", "only in eGRID", "only in OGE"],
    "blank in both",
)
coverage.pivot_table(
    index="sheet", columns="status", values="code", aggfunc="count", fill_value=0
).reindex(EGRID_SHEETS)

In [ ]:
# gaps in OGE's file by sheet and category
coverage[coverage["status"] == "only in eGRID"].pivot_table(
    index="category", columns="sheet", values="code", aggfunc="count", fill_value=0
).reindex(columns=EGRID_SHEETS, fill_value=0)

In [ ]:
# the columns of a sheet that are only in eGRID's file
sheet = "PLNT"
coverage.loc[
    (coverage["sheet"] == sheet) & (coverage["status"] == "only in eGRID"),
    ["code", "header", "category"],
]

In [ ]:
# columns of eGRID's file that are not in OGE's spec, e.g. columns that eGRID added
# or renamed in this year
{
    sheet: sorted(
        set(egrid[sheet].columns)
        - {code for code, _, _ in get_egrid_columns(sheet, year)}
    )
    for sheet in EGRID_SHEETS
}

## 2. Row matching

Rows are matched by `ROW_KEYS`. The comparisons below use only the rows in both files.

In [ ]:
def match_rows(sheet: str) -> pd.DataFrame:
    """Joins a sheet's OGE and eGRID rows on the sheet's keys, with "_oge" and
    "_egrid" added to the other columns and a "_merge" column saying which files have
    the row."""
    keys = ROW_KEYS[sheet] or ["row"]
    oge_table, egrid_table = oge[sheet], egrid[sheet]
    if not ROW_KEYS[sheet]:
        oge_table, egrid_table = oge_table.assign(row=0), egrid_table.assign(row=0)
    columns = sorted((set(oge_table.columns) | set(egrid_table.columns)) - set(keys))
    return oge_table.reindex(columns=keys + columns).merge(
        egrid_table.reindex(columns=keys + columns),
        on=keys,
        how="outer",
        suffixes=("_oge", "_egrid"),
        indicator=True,
    )


matched = {
    sheet: match_rows(sheet)
    for sheet in EGRID_SHEETS
    if not (oge[sheet].empty or egrid[sheet].empty)
}
pd.DataFrame(
    {
        sheet: table["_merge"]
        .value_counts()
        .rename(
            {
                "both": "in both",
                "left_only": "only in OGE",
                "right_only": "only in eGRID",
            }
        )
        for sheet, table in matched.items()
    }
).T

In [ ]:
# rows of a sheet that are only in one file, with their largest values first
sheet = "PLNT"
value = "PLNGENAN"
attributes = ["PNAME", "PSTATABB", "BACODE", "PLPRMFL"]
rows = matched[sheet]
for side, label in [("left_only", "oge"), ("right_only", "egrid")]:
    print(f"only in {label}:")
    display(
        rows.loc[
            rows["_merge"] == side,
            ROW_KEYS[sheet] + [f"{column}_{label}" for column in attributes + [value]],
        ]
        .sort_values(f"{value}_{label}", ascending=False, key=abs)
        .head(20)
    )

## 3. Number columns

For each column that is filled in both files, on the rows in both files:
- `oge_total` and `egrid_total` and their `total_ratio`, for columns that add up
  (not rates, shares or attributes)
- the number of rows where the column is blank in one file and filled in the other.
  eGRID leaves rates blank where the denominator is zero, so `blank only in OGE` counts
  are expected for those rates (see DAT-1068)
- the share of rows where both values are filled whose relative difference is within
  each of `TOLERANCES`

In [ ]:
def as_numbers(values: pd.Series) -> pd.Series | None:
    """Returns the values as numbers, or None if any filled value is not a
    number."""
    numbers = pd.to_numeric(values, errors="coerce")
    return None if (numbers.isna() & values.notna()).any() else numbers


def relative_difference(a: pd.Series, b: pd.Series) -> pd.Series:
    """Returns |a - b| / |b|, which is 0 where both are 0 and infinite where only b
    is 0."""
    difference = (a - b).abs()
    return (difference / b.abs()).where(b != 0, np.where(difference == 0, 0, np.inf))


def adds_up(header: str) -> bool:
    """Whether the totals of a column are meaningful, i.e. the column is a quantity
    and not a rate, share, or attribute."""
    unit = get_unit(header)
    return unit is not None and "/" not in unit and unit != "%"


comparisons = []
for sheet, rows in matched.items():
    both = rows[rows["_merge"] == "both"]
    filled = coverage[
        (coverage["sheet"] == sheet) & (coverage["status"] == "filled in both")
    ]
    for code, header, category in filled[["code", "header", "category"]].itertuples(
        index=False
    ):
        if code in SKIPPED_COLUMNS + ROW_KEYS[sheet]:
            continue
        a, b = as_numbers(both[f"{code}_oge"]), as_numbers(both[f"{code}_egrid"])
        if a is None or b is None:
            continue
        both_filled = a.notna() & b.notna()
        difference = relative_difference(a[both_filled], b[both_filled])
        comparison = {
            "sheet": sheet,
            "code": code,
            "header": header,
            "category": category,
            "rows": len(both),
            "oge_total": a.sum() if adds_up(header) else np.nan,
            "egrid_total": b.sum() if adds_up(header) else np.nan,
            "blank only in OGE": (a.isna() & b.notna()).sum(),
            "blank only in eGRID": (a.notna() & b.isna()).sum(),
        }
        for label, tolerance in TOLERANCES.items():
            comparison[label] = (difference <= tolerance).mean()
        comparison[">50%"] = (difference > 0.5).mean()
        comparisons.append(comparison)
number_comparison = pd.DataFrame(comparisons)
number_comparison["total_ratio"] = (
    number_comparison["oge_total"] / number_comparison["egrid_total"]
)
number_comparison = number_comparison[
    ["sheet", "code", "header", "category", "rows", "oge_total", "egrid_total"]
    + ["total_ratio", "blank only in OGE", "blank only in eGRID"]
    + list(TOLERANCES)
    + [">50%"]
]
number_comparison.groupby(["sheet", "category"], sort=False)[
    list(TOLERANCES) + [">50%"]
].median().round(2)

In [ ]:
# every number column of a sheet
sheet = "PLNT"
number_comparison[number_comparison["sheet"] == sheet].drop(columns="sheet").round(3)

In [ ]:
# the columns whose matched rows agree least, i.e. with the smallest share within 5%,
# on sheets with more than one row
number_comparison[number_comparison["rows"] > 1].sort_values("≤5%").head(30).round(3)

## 4. Code and text columns

Values are compared as numbers where both files' values are numbers (e.g. FIPS codes,
which OGE writes as text with leading zeros), and as text otherwise.

In [ ]:
def normalize(values: pd.Series) -> pd.Series:
    """Writes values as text without surrounding spaces, and numbers without a
    trailing ".0" or leading zeros."""
    numbers = pd.to_numeric(values, errors="coerce")
    text = values.astype("string").str.strip()
    return text.where(numbers.isna(), numbers.astype("Float64").astype("string"))


comparisons = []
for sheet, rows in matched.items():
    both = rows[rows["_merge"] == "both"]
    filled = coverage[
        (coverage["sheet"] == sheet) & (coverage["status"] == "filled in both")
    ]
    for code, header, category in filled[["code", "header", "category"]].itertuples(
        index=False
    ):
        if code in SKIPPED_COLUMNS + ROW_KEYS[sheet]:
            continue
        if (
            as_numbers(both[f"{code}_oge"]) is not None
            and as_numbers(both[f"{code}_egrid"]) is not None
        ):
            continue
        a, b = normalize(both[f"{code}_oge"]), normalize(both[f"{code}_egrid"])
        both_filled = a.notna() & b.notna()
        comparisons.append(
            {
                "sheet": sheet,
                "code": code,
                "header": header,
                "rows": len(both),
                "blank only in OGE": (a.isna() & b.notna()).sum(),
                "blank only in eGRID": (a.notna() & b.isna()).sum(),
                "match": (a[both_filled] == b[both_filled]).mean(),
            }
        )
code_comparison = pd.DataFrame(comparisons)
code_comparison.round(3)

In [ ]:
# the most common mismatched pairs of values of a column
sheet = "UNT"
code = "PRMVR"
both = matched[sheet][matched[sheet]["_merge"] == "both"]
pairs = pd.DataFrame(
    {"oge": normalize(both[f"{code}_oge"]), "egrid": normalize(both[f"{code}_egrid"])}
)
pairs[pairs["oge"].fillna("") != pairs["egrid"].fillna("")].value_counts(
    dropna=False
).head(20)

## 5. Drill-down

The matched rows of a sheet with the largest absolute differences in one number column,
with each file's values of related columns.

In [ ]:
sheet = "PLNT"
code = "PLNOXAN"
related = ["PNAME", "PSTATABB", "BACODE", "PLPRMFL", "PLNGENAN", "PLHTIANT"]

both = matched[sheet][matched[sheet]["_merge"] == "both"].copy()
both["difference"] = pd.to_numeric(both[f"{code}_oge"]) - pd.to_numeric(
    both[f"{code}_egrid"]
)
columns = ROW_KEYS[sheet] + [
    f"{column}_{label}"
    for column in [code] + related
    for label in ["oge", "egrid"]
    if f"{column}_{label}" in both
]
both.sort_values("difference", key=abs, ascending=False)[columns + ["difference"]].head(
    20
)

## 6. Known differences

These differences come from choices in OGE's methods or its eGRID file, so they are
expected:

- **Fuel categories** (DAT-1067): fuel columns (generation, resource mix, and rates by
  fuel type) use OGE's fuel categories under eGRID's headers. Gas is OGE's natural gas,
  oil is petroleum, other fossil is waste, and other is other plus storage. OGE assigns
  each subplant to its primary fuel, while eGRID assigns each plant to its primary fuel,
  so regional fuel totals and rates by fuel type differ even where plant totals match.
- **Rates with no denominator** (DAT-1068): OGE's eGRID file leaves rates blank where the
  denominator is zero or negative, as eGRID does. OGE's own results give a rate of zero
  in those cases.
- **CHP** (DAT-1069): the electric allocation factor, useful thermal output and power to
  heat ratio are blank until OGE outputs them. OGE's CHP adjustment also differs from
  eGRID's method.
- **NOx and SO2**: OGE fills and estimates emissions with its own methods, so annual NOx
  is about 6% higher and SO2 about 7% lower than eGRID for 2024.
- **CO2e**: OGE uses IPCC AR6 global warming potentials (CH4 27.9, N2O 273), and eGRID
  uses AR5 (CH4 28, N2O 265).
- **Units**: OGE's unit sheet has CEMS units only. eGRID also lists boilers and
  generators that are not in CEMS.
- **Plant IDs**: OGE's `ORISPL` is EIA's plant ID. eGRID uses EPA's plant ID for some
  plants whose IDs differ between EPA and EIA (e.g. EIA's 58215, Lee Combined Cycle, is
  EPA's 2709), so those plants show up as rows only in one file.
- **BA assignment**: OGE assigns some plants to different BAs than eGRID, which moves
  their values between rows of the BA sheet.
- **Not calculated yet**: the subregion, NERC region and grid loss sheets, regional Hg,
  and the nonbaseload columns are blank in OGE's file.